# Seaquest — Actor lambda-sweep + on-trajectory preview (Colab GPU)
We have only ever deployed the **greedy argmax-critic** policy (success_by_H=0.533 full-view). The original Contrastive RL (Eysenbach et al. 2022) deploys a **learned actor** pi(a|s,g) that maximises the critic, with an optional BC term — we never ran it. This trains that actor **faithfully** (reuses `train_actor.train_actor()` verbatim: objective `(1-lam)*E_pi[f(s,a,g)] + lam*log pi(a_teacher) + ent_coef*H(pi)`, critic frozen) and sweeps `lam in {0 pure-critic, 0.5 critic+BC, 1 pure-BC}`.

**Entropy bonus (`--ent-coef`, default 0.01).** The original actor is a continuous Gaussian policy with inherent entropy; the discrete categorical analog needs an explicit `H(pi)` term. Without it, `lam=0` maximising this near-flat critic (action-score range ~0.16) **collapses the softmax to a constant action** (vanishing gradient — verified on CPU/fp32, so it is an optimisation pathology, NOT an fp16 or code bug). `lam>0` is already protected by the BC term.

This notebook does the **env-free half**: train the 3 actors against the **full-view (oracle)** critic and measure on-trajectory agreement (a preview of the B/C/D verdict). The closed-loop success-rate comparison is a separate env step after the Docker rebuild.

**Needs:** GitHub TOKEN + Drive `raw_hf.zip` + Drive `critic_full_view.pt`.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 1. Clone repo at the committed actor-sweep script (main, latest).
TOKEN = 'PASTE_YOUR_GITHUB_TOKEN_HERE'
import os, subprocess
D = '/content/Goal-Conditioned-Confounded-MsPacman'
if not os.path.isdir(D):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git',D], check=True)
%cd /content/Goal-Conditioned-Confounded-MsPacman
!git pull -q && git log --oneline -1
assert os.path.exists('seaquest_ccrl/scripts/g0_actor_lambda_sweep.py'), 'sweep script missing — pull main'

In [ ]:
# 2. raw_hf from Drive (mount + unzip + auto-locate traj_*.npz).
import glob, os, zipfile
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/raw_hf.zip'         # <-- EDIT if elsewhere
assert os.path.exists(ZIP), f'zip not found at {ZIP}'
EXTRACT = '/content/raw_hf_extracted'
if not glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True):
    with zipfile.ZipFile(ZIP) as z: z.extractall(EXTRACT)
DATA_ROOT = os.path.dirname(glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True)[0])
n = len(glob.glob(DATA_ROOT + '/traj_*.npz'))
print('DATA_ROOT =', DATA_ROOT, '| trajectories:', n); assert n == 40

In [ ]:
# 3. Locate the PRE-TRAINED full-view (oracle) critic on Drive (auto-search MyDrive).
import glob
CRITIC = '/content/drive/MyDrive/critic_full_view.pt'    # <-- EDIT if elsewhere
if not os.path.exists(CRITIC):
    hits = glob.glob('/content/drive/MyDrive/**/critic_full_view.pt', recursive=True)
    assert hits, 'critic_full_view.pt not found on Drive — upload it (from artifacts/seaquest/seaquest_stage_g0_fullview_train/)'
    CRITIC = hits[0]
import torch
c = torch.load(CRITIC, map_location='cpu', weights_only=False)
print('CRITIC =', CRITIC, '| oracle =', c['oracle'], '| frame_stack =', c['cfg'].get('frame_stack'))
assert c['oracle'] is True and c['cfg'].get('frame_stack') == 4, 'expected the 4-frame full-view/oracle critic'

In [ ]:
# 4. SMOKE (one lambda, 20 steps) — verify the pipeline + JSON shape before the ~1.5h run.
!python -m seaquest_ccrl.scripts.g0_actor_lambda_sweep --critic-ckpt "$CRITIC" --root "$DATA_ROOT" --lams 0.5 --steps 20 --out-dir /content/actor_smoke
import json; print(json.dumps(json.load(open('/content/actor_smoke/on_trajectory_preview.json'))['lambda_sweep'], indent=2))

In [ ]:
# 5. FULL sweep: lam in {0.0, 0.5, 1.0} x 20k steps, entropy bonus 0.01 (GPU ~20-35 min each).
#    Watch the per-actor 'H' (entropy) in the log: ln(18)=2.89 uniform; it should NOT crash to 0.
!python -u -m seaquest_ccrl.scripts.g0_actor_lambda_sweep --critic-ckpt "$CRITIC" --root "$DATA_ROOT" --lams 0.0,0.5,1.0 --steps 20000 --ent-coef 0.01 --out-dir /content/actor_lambda_sweep

In [ ]:
# 6. Read the preview. Judge collapse by ENTROPY (ln18=2.89 uniform), not by argmax share.
#    Decision rule:
#      entropy_collapsed=True (entropy~0)  -> degenerate; if it persists even with ent_coef>0
#         the critic signal is too weak to drive an actor (A+C).
#      lam=0 healthy entropy + vdir_teacher >> chance + val_actor near val_argmax
#         -> the actor extracts coherent direction from the critic (B/D, pending env).
#      lam=1 (pure BC) = critic-FREE control ceiling; lam=0.5 beating it => critic adds value (D).
import json
r = json.load(open('/content/actor_lambda_sweep/on_trajectory_preview.json'))
print(f"critic={r['view']} steps={r['steps']} ent_coef={r['ent_coef']}  (ln18 uniform entropy = 2.89)\n")
hdr = ['lam','top1_teach','vdir_teach','top1_vsCrit','entropy','collapsed','val_actor','val_argmax']
print('{:>5} {:>11} {:>11} {:>12} {:>8} {:>10} {:>10} {:>10}'.format(*hdr))
for lam, d in r['lambda_sweep'].items():
    print('{:>5} {:>11.3f} {:>11.3f} {:>12.3f} {:>8.3f} {:>10} {:>10.3f} {:>10.3f}'.format(
        lam, d['top1_agree_teacher'], d['vdir_agree_teacher'], d['top1_agree_argmax_critic'],
        d['actor_entropy_nats'], str(d['entropy_collapsed']), d['critic_value_actor_Epi_f'],
        d['critic_value_argmax_f']))

In [ ]:
# 7. Persist preview JSON + the 3 actor checkpoints to Drive + download zip.
import shutil, glob, os
OUT = '/content/actor_sweep_out'; os.makedirs(OUT, exist_ok=True)
shutil.copy('/content/actor_lambda_sweep/on_trajectory_preview.json', OUT)
for p in glob.glob('/content/actor_lambda_sweep/lam*/actor_oracle.pt'):
    lam = os.path.basename(os.path.dirname(p))            # lam0.00 / lam0.50 / lam1.00
    shutil.copy(p, os.path.join(OUT, f'actor_oracle_{lam}.pt'))
shutil.make_archive('seaquest_actor_lambda_sweep', 'zip', OUT)
try:
    from google.colab import drive; drive.mount('/content/drive')
    shutil.copy('seaquest_actor_lambda_sweep.zip', '/content/drive/MyDrive/')
    print('copied zip to Drive')
except Exception as e:
    print('drive copy skipped:', e)
from google.colab import files; files.download('seaquest_actor_lambda_sweep.zip')

## Next — closed-loop verdict (env, after Docker rebuild)
This preview is **env-free**. The actual B/C/D answer is the closed-loop success-rate: run `g0_closed_loop_eval.py` with each `actor_oracle_lam*.pt` (`--actor-ckpt`) on the **same anchor protocol** as the 0.533 argmax baseline, and compare success_by_H / success_at_H across argmax-critic vs lam=0/0.5/1.0 vs random. Paste `on_trajectory_preview.json` back for the preview read first.